In [13]:
import numpy as np
import tensorflow as tf
import keras
from keras import layers, models

In [14]:
sentences = "Input Sequences"
outputs = ["output Sequences"]

In [15]:
MAX_LEN = 100
VOCAB_SIZE = 15000
EMBEDDING_DIM = 512
NUM_HEADS = 8

vectorizer = layers.TextVectorization(
    max_tokens = VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_LEN
)
vectorizer.adapt(sentences)
vocab = vectorizer.get_vocabulary()

output_vectorizer = layers.TextVectorization(
    max_tokens = VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=MAX_LEN
)
output_vectorizer.adapt(outputs)
output_vocab = output_vectorizer.get_vocabulary()

In [18]:
class TransformerEncoderLayer(layers.Layer):

  def __init__(self, d_model, num_heads, dff, dropout_rate=0.1):
    super().__init__()
    self.mha = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=d_model // num_heads
    )
    self.ffn = keras.Sequential([
        layers.Dense(dff, activation="relu"),
        layers.Dense(d_model),
    ])

    self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
    self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
    self.dropout1 = layers.Dropout(dropout_rate)
    self.dropout2 = layers.Dropout(dropout_rate)

  def call(self, x, training=False, mask=None):
    # 1. Self-Attention + Add & Norm
    attn_output = self.mha(query=x, value=x, key=x, attention_mask=mask)
    attn_output = self.dropout1(attn_output, training=training)
    out1 = self.layernorm1(x + attn_output)

    # 2. Feed-Forward Network + Add & Norm
    ffn_output = self.ffn(out1)
    ffn_output = self.dropout2(ffn_output, training=training)
    return self.layernorm2(out1 + ffn_output)

In [19]:
def positional_encoding(length, depth):
  depth = depth / 2
  positions = np.arange(length)[:, np.newaxis]
  depths = np.arange(depth)[np.newaxis, :] / depth

  angle_rates = 1 / (10000**depths)
  angle_rads = positions * angle_rates

  pos_encoding = np.concatenate(
      [np.sin(angle_rads), np.cos(angle_rads)], axis=-1
  )
  return tf.cast(pos_encoding, dtype=tf.float32)

In [20]:
class TransformerEncoder(layers.Layer):

  def __init__(
      self,
      num_layers,
      d_model,
      num_heads,
      dff,
      input_vocab_size,
      max_seq_len=512,
      dropout_rate=0.1,
  ):
    super().__init__()
    self.d_model = d_model
    self.num_layers = num_layers

    self.embedding = layers.Embedding(input_vocab_size, d_model)
    self.pos_encoding = positional_encoding(
        length=max_seq_len, depth=d_model
    )

    self.enc_layers = [
        TransformerEncoderLayer(d_model, num_heads, dff, dropout_rate)
        for _ in range(num_layers)
    ]
    self.dropout = layers.Dropout(dropout_rate)

  def call(self, x, training=False, mask=None):
    seq_len = tf.shape(x)[1]

    # Scale embeddings and add positional encoding
    x = self.embedding(x)
    x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
    x += self.pos_encoding[tf.newaxis, :seq_len, :]

    x = self.dropout(x, training=training)

    # Pass through stacked encoder layers
    for i in range(self.num_layers):
      x = self.enc_layers[i](x, training=training, mask=mask)

    return x

In [21]:
class TransformerDecoderLayer(layers.Layer):

  def __init__(self, d_model, num_heads, dff, dropout_rate=0.1):
    super().__init__()
    # 1. Masked (Causal) Self-Attention
    self.causal_mha = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=d_model // num_heads
    )
    # 2. Cross-Attention (attends to Encoder output)
    self.cross_mha = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=d_model // num_heads
    )
    # 3. Feed-Forward Network
    self.ffn = keras.Sequential([
        layers.Dense(dff, activation="relu"),
        layers.Dense(d_model),
    ])

    self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
    self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
    self.layernorm3 = layers.LayerNormalization(epsilon=1e-6)

    self.dropout1 = layers.Dropout(dropout_rate)
    self.dropout2 = layers.Dropout(dropout_rate)
    self.dropout3 = layers.Dropout(dropout_rate)

  def call(
      self,
      x,
      enc_output,
      training=False,
      look_ahead_mask=None,
      padding_mask=None,
  ):
    # 1. Causal Self-Attention + Add & Norm
    # use_causal_mask=True automatically prevents attending to future tokens
    attn1 = self.causal_mha(
        query=x,
        value=x,
        key=x,
        attention_mask=look_ahead_mask,
        use_causal_mask=True,
    )
    attn1 = self.dropout1(attn1, training=training)
    out1 = self.layernorm1(x + attn1)

    # 2. Cross-Attention + Add & Norm (Query = Decoder, Key/Value = Encoder)
    attn2 = self.cross_mha(
        query=out1,
        value=enc_output,
        key=enc_output,
        attention_mask=padding_mask,
    )
    attn2 = self.dropout2(attn2, training=training)
    out2 = self.layernorm2(out1 + attn2)

    # 3. Feed-Forward Network + Add & Norm
    ffn_output = self.ffn(out2)
    ffn_output = self.dropout3(ffn_output, training=training)
    return self.layernorm3(out2 + ffn_output)

In [22]:
class TransformerDecoder(layers.Layer):

  def __init__(
      self,
      num_layers,
      d_model,
      num_heads,
      dff,
      target_vocab_size,
      max_seq_len=512,
      dropout_rate=0.1,
  ):
    super().__init__()
    self.d_model = d_model
    self.num_layers = num_layers

    self.embedding = layers.Embedding(target_vocab_size, d_model)
    self.pos_encoding = positional_encoding(
        length=max_seq_len, depth=d_model
    )

    self.dec_layers = [
        TransformerDecoderLayer(d_model, num_heads, dff, dropout_rate)
        for _ in range(num_layers)
    ]
    self.dropout = layers.Dropout(dropout_rate)

  def call(
      self,
      x,
      enc_output,
      training=False,
      look_ahead_mask=None,
      padding_mask=None,
  ):
    seq_len = tf.shape(x)[1]

    x = self.embedding(x)
    x *= tf.math.sqrt(tf.cast(self.d_model, tf.float32))
    x += self.pos_encoding[tf.newaxis, :seq_len, :]

    x = self.dropout(x, training=training)

    for i in range(self.num_layers):
      x = self.dec_layers[i](
          x,
          enc_output,
          training=training,
          look_ahead_mask=look_ahead_mask,
          padding_mask=padding_mask,
      )

    return x

In [23]:
class Transformer(keras.Model):

  def __init__(
      self,
      num_layers,
      d_model,
      num_heads,
      dff,
      input_vocab_size,
      target_vocab_size,
      max_seq_len=512,
      dropout_rate=0.1,
  ):
    super().__init__()
    self.encoder = TransformerEncoder(
        num_layers,
        d_model,
        num_heads,
        dff,
        input_vocab_size,
        max_seq_len,
        dropout_rate,
    )
    self.decoder = TransformerDecoder(
        num_layers,
        d_model,
        num_heads,
        dff,
        target_vocab_size,
        max_seq_len,
        dropout_rate,
    )
    # Projects final representation to vocabulary size logits
    self.final_layer = layers.Dense(target_vocab_size)

  def call(self, inputs, training=False):
    # Expects a tuple of (source_tokens, target_tokens)
    context, x = inputs

    # Create padding masks where token ID == 0 (1 = valid token, 0 = pad)
    enc_padding_mask = self.create_padding_mask(context)
    dec_padding_mask = self.create_padding_mask(context)
    look_ahead_mask = self.create_padding_mask(x)

    # 1. Encode source sequence
    enc_output = self.encoder(
        context, training=training, mask=enc_padding_mask
    )

    # 2. Decode target sequence using encoder output
    dec_output = self.decoder(
        x,
        enc_output,
        training=training,
        look_ahead_mask=look_ahead_mask,
        padding_mask=dec_padding_mask,
    )

    # 3. Final linear projection to logits
    logits = self.final_layer(dec_output)

    # Drop keras mask to avoid propagating it to loss functions
    try:
      del logits._keras_mask
    except AttributeError:
      pass

    return logits

  def create_padding_mask(self, seq):
    # Creates boolean mask of shape (batch_size, 1, seq_len)
    # True = keep token, False = mask out (pad token 0)
    mask = tf.math.not_equal(seq, 0)
    return mask[:, tf.newaxis, :]


In [25]:
transformer = Transformer(
    num_layers=4,
    d_model=128,
    num_heads=8,
    dff=512,
    input_vocab_size=10000,
    target_vocab_size=8000,
)